In [3]:
pip install statsmodels

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB 435.7 kB/s eta 0:00:22
   ---------------------------------------- 0.1/9.5 MB 653.6 kB/s eta 0:00:15
    --------------------------------------- 0.2/9.5 MB 1.0 MB/s eta 0:00:10
   - -------------------------------------- 0.3/9.5 MB 1.4 MB/s eta 0:00:07
   - -------------------------------------- 0.4/9.5 MB 1.8 MB/s eta 0:00:06
   -- ------------------------------------- 0.7/9.5 MB 2.3 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/9.5 MB 2.2 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/9.5 MB 2.2 MB/s eta 0:00:04
   --- ------------------------------------ 0.7/9.5 MB 1.8 MB/s eta 0:00:05
   --- ------------------------------------ 0.8/9.5 MB 1.6 MB/s eta 0:00:06
   ----- ---------------------------------- 1.2/9.5 MB 2.5 MB/s eta 0:00:04
   ----- -------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import & Veri

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from codecarbon import OfflineEmissionsTracker
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time, os, copy, json

os.makedirs("results", exist_ok=True)

# Sunspot verisi — sklearn'den veya CSV'den
import statsmodels.api as sm
data = sm.datasets.sunspots.load_pandas().data
values = data['SUNACTIVITY'].values.astype(float).reshape(-1, 1)
print(f"Veri boyutu: {len(data)}")
print(data.head())
# Eğer bu çalışmazsa alternatif:
# data = pd.read_csv("sunspot.csv")

print(f"Veri boyutu: {len(data)}")
print(data.head())

Veri boyutu: 309
     YEAR  SUNACTIVITY
0  1700.0          5.0
1  1701.0         11.0
2  1702.0         16.0
3  1703.0         23.0
4  1704.0         36.0
Veri boyutu: 309
     YEAR  SUNACTIVITY
0  1700.0          5.0
1  1701.0         11.0
2  1702.0         16.0
3  1703.0         23.0
4  1704.0         36.0


In [5]:
# Veri Hazırlığı

scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

# Train/test split — son 20% test
split = int(len(scaled) * 0.8)
train_data = scaled[:split]
test_data = scaled[split:]

# Sequence oluştur
SEQ_LEN = 24  # 24 ay = 2 yıl geriye bak

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return torch.FloatTensor(np.array(X)), torch.FloatTensor(np.array(y))

X_train, y_train = create_sequences(train_data, SEQ_LEN)
X_test, y_test = create_sequences(test_data, SEQ_LEN)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: torch.Size([223, 24, 1]), Test: torch.Size([38, 24, 1])


In [6]:
# LSTM Model Tanımı & Yardımcı Fonksiyonlar
class SunspotLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # son time step
        return out

def train_model(model, X_train, y_train, epochs, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    losses = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} — Loss: {loss.item():.6f}")
    return losses

def evaluate_model(model, X_test, y_test, scaler):
    model.eval()
    with torch.no_grad():
        preds = model(X_test).numpy()
    
    # Inverse transform
    preds_inv = scaler.inverse_transform(preds)
    actual_inv = scaler.inverse_transform(y_test.numpy())
    
    mae = mean_absolute_error(actual_inv, preds_inv)
    rmse = np.sqrt(mean_squared_error(actual_inv, preds_inv))
    return mae, rmse, preds_inv

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def measure_inference(model, X_test, n_runs=50):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model(X_test)
            times.append(time.perf_counter() - t0)
    return np.mean(times)

results = []

In [5]:
# DENEY 1: Baseline LSTM (hidden=256, 50 epoch)

print("=" * 60)
print("DENEY 1: Baseline LSTM (hidden=256, 50 epoch)")
print("=" * 60)

model_base = SunspotLSTM(hidden_size=256, num_layers=2)

tracker = OfflineEmissionsTracker(
    project_name="lstm_baseline",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

losses_base = train_model(model_base, X_train, y_train, epochs=50)

emissions_base = tracker.stop()

mae_base, rmse_base, _ = evaluate_model(model_base, X_test, y_test, scaler)
inf_time_base = measure_inference(model_base, X_test)

results.append({
    "model": "LSTM",
    "experiment": "Baseline (hidden=256, 50ep)",
    "mae": round(mae_base, 4),
    "rmse": round(rmse_base, 4),
    "model_size_mb": round(get_model_size_mb(model_base), 4),
    "parameters": count_params(model_base),
    "inference_time_s": round(inf_time_base, 6),
    "energy_kwh": emissions_base if emissions_base else 0,
    "epochs_trained": 50
})

baseline_state = copy.deepcopy(model_base.state_dict())

print(f"\nMAE: {mae_base:.4f}, RMSE: {rmse_base:.4f}")
print(f"Boyut: {get_model_size_mb(model_base):.4f} MB, Params: {count_params(model_base):,}")
print(f"Enerji: {emissions_base} kWh")

[codecarbon INFO @ 20:00:13] offline tracker init
[codecarbon WARNING @ 20:00:13] Multiple instances of codecarbon are allowed to run at the same time.


DENEY 1: Baseline LSTM (hidden=256, 50 epoch)
  Epoch 10/50 — Loss: 0.035573
  Epoch 20/50 — Loss: 0.033267
  Epoch 30/50 — Loss: 0.032066
  Epoch 40/50 — Loss: 0.030986
  Epoch 50/50 — Loss: 0.029145

MAE: 38.6810, RMSE: 49.7628
Boyut: 3.0205 MB, Params: 791,809
Enerji: 3.555963505572057e-05 kWh


In [6]:
# DENEY 2: Dynamic Quantization INT8
print("=" * 60)
print("DENEY 2: Dynamic Quantization (INT8)")
print("=" * 60)

model_q = SunspotLSTM(hidden_size=256, num_layers=2)
model_q.load_state_dict(baseline_state)

tracker = OfflineEmissionsTracker(
    project_name="lstm_quantized",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

model_q_int8 = torch.quantization.quantize_dynamic(
    model_q,
    {nn.LSTM, nn.Linear},
    dtype=torch.qint8
)

emissions_q = tracker.stop()

# Quantized modelin dosya boyutunu ölç
torch.save(model_q_int8.state_dict(), "results/lstm_quantized.pt")
quant_file_size = os.path.getsize("results/lstm_quantized.pt") / (1024 ** 2)

mae_q, rmse_q, _ = evaluate_model(model_q_int8, X_test, y_test, scaler)
inf_time_q = measure_inference(model_q_int8, X_test)

results.append({
    "model": "LSTM",
    "experiment": "Dynamic Quantization INT8",
    "mae": round(mae_q, 4),
    "rmse": round(rmse_q, 4),
    "model_size_mb": round(quant_file_size, 4),
    "parameters": count_params(model_q),
    "inference_time_s": round(inf_time_q, 6),
    "energy_kwh": emissions_q if emissions_q else 0,
    "epochs_trained": 0
})

print(f"MAE: {mae_q:.4f}, RMSE: {rmse_q:.4f}")
print(f"Dosya boyutu: {quant_file_size:.4f} MB (baseline: {get_model_size_mb(model_base):.4f} MB)")
print(f"Inference: {inf_time_q:.6f}s vs baseline: {inf_time_base:.6f}s")

DENEY 2: Dynamic Quantization (INT8)


C:\Users\bahar\AppData\Local\Temp\ipykernel_22564\3361576389.py:19: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_q_int8 = torch.quantization.quantize_dynamic(


MAE: 38.6879, RMSE: 49.7719
Dosya boyutu: 0.7722 MB (baseline: 3.0205 MB)
Inference: 0.025149s vs baseline: 0.009664s


In [7]:
# DENEY 3: Early Stopping (max 100, patience 10)

print("=" * 60)
print("DENEY 3: Early Stopping (max 100 epoch, patience 10)")
print("=" * 60)

model_es = SunspotLSTM(hidden_size=256, num_layers=2)
optimizer_es = torch.optim.Adam(model_es.parameters(), lr=0.001)
criterion = nn.MSELoss()

tracker = OfflineEmissionsTracker(
    project_name="lstm_early_stopping",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

best_val_loss = float("inf")
patience = 10
patience_counter = 0
best_state_es = None
stopped_epoch = 0
es_train_losses = []
es_val_losses = []

for epoch in range(100):
    model_es.train()
    optimizer_es.zero_grad()
    output = model_es(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer_es.step()
    es_train_losses.append(loss.item())
    
    # Validation loss
    model_es.eval()
    with torch.no_grad():
        val_pred = model_es(X_test)
        val_loss = criterion(val_pred, y_test).item()
    es_val_losses.append(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/100 — Train Loss: {loss.item():.6f}, Val Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state_es = copy.deepcopy(model_es.state_dict())
        stopped_epoch = epoch + 1
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  >>> Early stopping at epoch {epoch+1}, best epoch: {stopped_epoch}")
            break

emissions_es = tracker.stop()

model_es.load_state_dict(best_state_es)
mae_es, rmse_es, _ = evaluate_model(model_es, X_test, y_test, scaler)
inf_time_es = measure_inference(model_es, X_test)

results.append({
    "model": "LSTM",
    "experiment": f"Early Stopping (best ep {stopped_epoch})",
    "mae": round(mae_es, 4),
    "rmse": round(rmse_es, 4),
    "model_size_mb": round(get_model_size_mb(model_es), 4),
    "parameters": count_params(model_es),
    "inference_time_s": round(inf_time_es, 6),
    "energy_kwh": emissions_es if emissions_es else 0,
    "epochs_trained": stopped_epoch
})

print(f"\nBest epoch: {stopped_epoch}, MAE: {mae_es:.4f}, RMSE: {rmse_es:.4f}")
print(f"Enerji: {emissions_es} kWh")

DENEY 3: Early Stopping (max 100 epoch, patience 10)
  Epoch 10/100 — Train Loss: 0.035479, Val Loss: 0.093768
  >>> Early stopping at epoch 15, best epoch: 5

Best epoch: 5, MAE: 42.4005, RMSE: 48.9178
Enerji: 1.0821187893237244e-05 kWh


In [8]:
#  DENEY 4: Küçük LSTM (hidden=128)

print("=" * 60)
print("DENEY 4: Küçük LSTM (hidden=128, 50 epoch)")
print("=" * 60)

model_small = SunspotLSTM(hidden_size=128, num_layers=2)

tracker = OfflineEmissionsTracker(
    project_name="lstm_small",
    output_dir="results",
    country_iso_code="TUR",
    log_level="error",
    tracking_mode="machine",
    allow_multiple_runs=True
)
tracker.start()

losses_small = train_model(model_small, X_train, y_train, epochs=50)

emissions_small = tracker.stop()

mae_small, rmse_small, _ = evaluate_model(model_small, X_test, y_test, scaler)
inf_time_small = measure_inference(model_small, X_test)

results.append({
    "model": "LSTM",
    "experiment": "Küçük Model (hidden=128, 50ep)",
    "mae": round(mae_small, 4),
    "rmse": round(rmse_small, 4),
    "model_size_mb": round(get_model_size_mb(model_small), 4),
    "parameters": count_params(model_small),
    "inference_time_s": round(inf_time_small, 6),
    "energy_kwh": emissions_small if emissions_small else 0,
    "epochs_trained": 50
})

print(f"\nMAE: {mae_small:.4f}, RMSE: {rmse_small:.4f}")
print(f"Boyut: {get_model_size_mb(model_small):.4f} MB (baseline: {get_model_size_mb(model_base):.4f} MB)")
print(f"Params: {count_params(model_small):,} (baseline: {count_params(model_base):,})")
print(f"Enerji: {emissions_small} kWh")

DENEY 4: Küçük LSTM (hidden=128, 50 epoch)
  Epoch 10/50 — Loss: 0.043579
  Epoch 20/50 — Loss: 0.036057
  Epoch 30/50 — Loss: 0.033300
  Epoch 40/50 — Loss: 0.032435
  Epoch 50/50 — Loss: 0.031714

MAE: 41.3041, RMSE: 51.4007
Boyut: 0.7603 MB (baseline: 3.0205 MB)
Params: 199,297 (baseline: 791,809)
Enerji: 1.5229971063958071e-05 kWh


In [9]:
# LSTM Sonuçları Kaydet

df_lstm = pd.DataFrame(results)
df_lstm.to_csv("results/lstm_results.csv", index=False)
print(df_lstm.to_string(index=False))

model                     experiment     mae    rmse  model_size_mb  parameters  inference_time_s  energy_kwh  epochs_trained
 LSTM    Baseline (hidden=256, 50ep) 38.6810 49.7628         3.0205      791809          0.009664    0.000036              50
 LSTM      Dynamic Quantization INT8 38.6879 49.7719         0.7722      791809          0.025149    0.000003               0
 LSTM     Early Stopping (best ep 5) 42.4005 48.9178         3.0205      791809          0.009856    0.000011               5
 LSTM Küçük Model (hidden=128, 50ep) 41.3041 51.4007         0.7603      199297          0.003798    0.000015              50


In [7]:
# Tüm Sonuçları Birleştir
df_mobilenet = pd.read_csv("../results/mobilenet_results.csv")
df_distilbert = pd.read_csv("results/distilbert_results.csv")

# Normalize et — ortak sütunlar
def normalize_mobilenet(df):
    return pd.DataFrame({
        'model': df['Model'],
        'experiment': df['Teknik'],
        'accuracy': df['Accuracy'],
        'f1_score': df['F1'],
        'model_size_mb': df['Model Boyutu (MB)'],
        'energy_kwh': df['Enerji (kWh)'],
        'training_time_s': df['Eğitim Süresi (sn)'],
        'co2_kg': df['CO2 (kg)'],
        'parameters': None,
        'inference_time_s': None,
        'mae': None,
        'rmse': None,
    })

def normalize_distilbert(df):
    return pd.DataFrame({
        'model': df['model'],
        'experiment': df['experiment'],
        'accuracy': df['accuracy'],
        'f1_score': df['f1_score'],
        'model_size_mb': df['model_size_mb'],
        'energy_kwh': df['energy_kwh'],
        'training_time_s': None,
        'co2_kg': None,
        'parameters': df['parameters'],
        'inference_time_s': df['inference_time_s'],
        'mae': None,
        'rmse': None,
    })

def normalize_lstm(df):
    return pd.DataFrame({
        'model': df['model'],
        'experiment': df['experiment'],
        'accuracy': None,  # LSTM regresyon, accuracy yok
        'f1_score': None,
        'model_size_mb': df['model_size_mb'],
        'energy_kwh': df['energy_kwh'],
        'training_time_s': None,
        'co2_kg': None,
        'parameters': df['parameters'],
        'inference_time_s': df['inference_time_s'],
        'mae': df['mae'],
        'rmse': df['rmse'],
    })

df_all = pd.concat([
    normalize_mobilenet(df_mobilenet),
    normalize_distilbert(df_distilbert),
    normalize_lstm(df_lstm)
], ignore_index=True)

df_all.to_csv("results/all_results.csv", index=False)
print(f"Toplam {len(df_all)} deney kaydedildi.")
print(df_all.to_string(index=False))
import os
print("CSV yazıldığı yol:", os.path.abspath("results/all_results.csv"))

NameError: name 'df_lstm' is not defined

In [12]:
# Metadata
metadata = {
    "lstm_experiments": {
        "dataset": "NASA Sunspot Time Series",
        "total_data_points": len(values),
        "train_size": len(train_data),
        "test_size": len(test_data),
        "sequence_length": SEQ_LEN,
        "baseline_hidden_size": 256,
        "small_hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.2,
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "loss_function": "MSELoss",
        "quantization": "Dynamic INT8 (nn.LSTM + nn.Linear)",
        "early_stopping": {"max_epochs": 100, "patience": 10, "monitor": "val_loss"},
        "device": "CPU",
        "pytorch_version": torch.__version__,
        "codecarbon": {"mode": "offline", "country": "TUR", "gpu_tracking": False}
    }
}

with open("results/lstm_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Metadata kaydedildi: results/lstm_metadata.json")

Metadata kaydedildi: results/lstm_metadata.json
